This phase resolves the configured pair set and acquires every PMID and title returned by each pathogen–host query. PubMed queries include accepted pathogen names, mapped nomenclatural aliases, and taxon-checked ICTV virus names and abbreviations that uniquely map to one pathogen. Raw isolate/strain labels, unresolved names, and cross-taxon homonyms remain in taxonomy provenance but are excluded from search. The pair-search cache is the source corpus for the host-specific pathogen yes/no screen in Step 4. The configurable sample cap controls that screen; the pair searches themselves retain all results.

**Saved outputs are historical.** They were produced before host-mention expansion was added; rerun notebook 00, then this notebook, to refresh the query previews and acquisition results. 

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    raise RuntimeError('Open this notebook from the V4 repository')
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'notebooks'))
from _shared import configure_notebook, configured_run_id, log_step, ncbi_contact_email
RUN_ID = configured_run_id(PROJECT_ROOT)
ROOT, LOG, LOG_PATH = configure_notebook('03_acquire_pair_pubmed_records', RUN_ID)
print('Project root:', ROOT, '| log:', LOG_PATH)

2026-11-03 00:38 INFO Notebook session started at 2026-11-03T00:38:53.761638+00:00


2026-11-03 00:38 INFO Project root: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4


2026-11-03 00:38 INFO Run log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/pilot-v1/03_acquire_pair_pubmed_records.log


Project root: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4 | log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/pilot-v1/03_acquire_pair_pubmed_records.log


## Step 1 · Load configuration and taxonomy references

In [2]:
import yaml, pandas as pd
from IPython.display import display
from pathogen_atlas_v4.runs import resolve_run, save_resolved_run
from pathogen_atlas_v4.pubmed import build_pair_query
from pathogen_atlas_v4.pubmed_client import pubmed_aliases_by_pathogen
CONFIG = yaml.safe_load((ROOT / 'configs/pilot_v1.yaml').read_text())
HOST_SEARCH_EXPANSIONS = CONFIG['acquisition'].get('host_search_expansions', {})
HOST_EXPANSION_HASH = CONFIG['acquisition'].get('host_expansion_hash', '')
if CONFIG['acquisition'].get('host_expansion_enabled', False):
    configured_host_ids = set(map(str, CONFIG['run'].get('host_ids', [])))
    missing_expansion_hosts = sorted(configured_host_ids - set(HOST_SEARCH_EXPANSIONS))
    if missing_expansion_hosts:
        raise ValueError(f'Host expansion is enabled but terms are missing for {missing_expansion_hosts}; rerun notebook 00_configure_run.ipynb first.')
TAXONOMY_ROOT = sorted((ROOT / 'data/v4/staging').glob('*/taxonomy'))[-1]
RUN_TAXONOMY_ROOT = ROOT / 'data/v4/runs' / CONFIG['run']['run_id'] / 'taxonomy'
if not (RUN_TAXONOMY_ROOT / 'pathogens.parquet').exists():
    raise FileNotFoundError('Run taxonomy was not resolved; run 00_configure_run.ipynb first.')
hosts = pd.read_parquet(RUN_TAXONOMY_ROOT / 'hosts.parquet')
pathogens = pd.read_parquet(RUN_TAXONOMY_ROOT / 'pathogens.parquet')
ALIASES = pd.read_parquet(RUN_TAXONOMY_ROOT / 'pathogen_aliases.parquet')
ALIASES_BY_PATHOGEN, ALIAS_FILTER_COUNTS = pubmed_aliases_by_pathogen(pathogens, ALIASES)
LOG.info('PubMed alias safeguard: %s', ALIAS_FILTER_COUNTS)
LOG.info('Inputs loaded: hosts=%d pathogens=%d config=%s', len(hosts), len(pathogens), CONFIG['run']['run_id'])
display(CONFIG['run'])

2026-11-03 00:38 INFO PubMed alias safeguard: {'aliases_total': 233550, 'aliases_missing_name': 0, 'aliases_unmapped': 21291, 'aliases_non_nomenclatural': 54478, 'aliases_shared_across_pathogens': 3267, 'aliases_vmr_records_checked': 36743, 'aliases_vmr_terms_extracted': 65410, 'aliases_searchable': 29251}


2026-11-03 00:38 INFO Inputs loaded: hosts=9 pathogens=142201 config=pilot-v1


{'run_id': 'pilot-v1',
 'config_version': 'pilot-config-v1',
 'selection_mode': 'full',
 'selection_seed': 'v4-pilot-v1',
 'host_ids': ['ncbi:4530'],
 'pathogen_ids': [],
 'pair_generation': 'cartesian_product',
 'pathogen_groups': ['viruses']}

## Step 2 · Resolve the exact pair set

In [3]:
host_groups = {key: group.host_id.astype(str).tolist() for key, group in hosts.groupby('host_group')}
pathogen_groups = {key: group.pathogen_id.astype(str).tolist() for key, group in pathogens.groupby('pathogen_group')}
resolved = resolve_run(CONFIG, known_host_ids=set(hosts.host_id.astype(str)),
                       known_pathogen_ids=set(pathogens.pathogen_id.astype(str)),
                       host_group_ids=host_groups, pathogen_group_ids=pathogen_groups)
RESOLVED_PATH = ROOT / 'data/v4/runs' / resolved.run_id / 'resolved_config.json'
save_resolved_run(CONFIG, resolved, RESOLVED_PATH)
print('Run:', resolved.run_id, '| generation rule:', resolved.pair_generation)
print('Hosts:', len(resolved.host_ids), '| pathogens:', len(resolved.pathogen_ids), '| pairs:', len(resolved.pairs))
print('Resolved configuration:', RESOLVED_PATH)
LOG.info('Resolved and saved run: hosts=%d pathogens=%d pairs=%d hash=%s', len(resolved.host_ids), len(resolved.pathogen_ids), len(resolved.pairs), resolved.resolved_config_hash)

2026-11-03 00:38 INFO Resolved and saved run: hosts=1 pathogens=17554 pairs=17554 hash=01476ff0de0ca2a3655fa082efb5a743993e9c4a1a4c3c425ba48230d5a0d3ed


Run: pilot-v1 | generation rule: cartesian_product
Hosts: 1 | pathogens: 17554 | pairs: 17554
Resolved configuration: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/runs/pilot-v1/resolved_config.json


## Step 3 · Build and inspect pair-aware query text

In [4]:
host_rows = hosts.set_index('host_id')
pathogen_rows = pathogens.set_index('pathogen_id')
PREVIEW_ROWS = []
for pathogen_id, host_id in resolved.pairs[:10]:
    pathogen = pathogen_rows.loc[pathogen_id]
    host = host_rows.loc[host_id]
    host_aliases = host.get('aliases', [])
    if hasattr(host_aliases, 'tolist'):
        host_aliases = host_aliases.tolist()
    expansion = HOST_SEARCH_EXPANSIONS.get(str(host_id), {})
    host_mention_terms = [item['term'] for item in expansion.get('terms', [])]
    query = build_pair_query([pathogen.accepted_name, *ALIASES_BY_PATHOGEN.get(str(pathogen_id), [])],
                             [host.accepted_name, *(host_aliases or [])],
                             host_mention_terms=host_mention_terms, host_mesh_terms=expansion.get('mesh_terms', []),
                             strategy_version=CONFIG['acquisition']['query_strategy_version'])
    PREVIEW_ROWS.append({'pathogen_id': pathogen_id, 'pathogen': pathogen.accepted_name,
                         'host_id': host_id, 'host': host.accepted_name,
                         'host_cue_count': len(host_mention_terms), 'host_mesh_terms': expansion.get('mesh_terms', []), **query})
display(pd.DataFrame(PREVIEW_ROWS))
LOG.info('Generated %d preview queries; full acquisition covers all %d pairs with strategy %s', len(PREVIEW_ROWS), len(resolved.pairs), CONFIG['acquisition']['query_strategy_version'])


,pathogen_id,pathogen,host_id,host,query,query_hash,query_version
0,viruses:868bbe896ed10f2211ce,Aalivirus apekidu,ncbi:4530,Oryza sativa,"(""aalivirus A1""[Title/Abstract] OR ""Aalivirus ...",b3ba960f37db38466307f3f6b87aa53adaffe082c003fb...,pathogen-host-ictv-vmr-virus-names-title-abstr...
1,viruses:b5cb6933e3ce1c093121,Aarhusvirus dagda,ncbi:4530,Oryza sativa,"(""Aarhusvirus dagda""[Title/Abstract] OR ""Dicke...",89ccdc283171240e5fbe8aad96eb6d5535d0c9421ea513...,pathogen-host-ictv-vmr-virus-names-title-abstr...
2,viruses:1008e449aa6f64fbd9b5,Aarhusvirus katbat,ncbi:4530,Oryza sativa,"(""Aarhusvirus katbat""[Title/Abstract] OR ""Dick...",45eb1d4dd5906293b7d95e9803d193855edc30a7e0bdd9...,pathogen-host-ictv-vmr-virus-names-title-abstr...
3,viruses:e74b098fac456c8aac76,Aarhusvirus luksen,ncbi:4530,Oryza sativa,"(""Aarhusvirus luksen""[Title/Abstract] OR ""Dick...",2fa08207ad862174ea2c3777e0bf21c3e65cff7678c67c...,pathogen-host-ictv-vmr-virus-names-title-abstr...
4,viruses:34019f11cbba198868d0,Aarhusvirus mysterion,ncbi:4530,Oryza sativa,"(""Aarhusvirus mysterion""[Title/Abstract] OR ""D...",3ab8ad858e531d7bb81a9a3ab08741c1dd8903c2df325d...,pathogen-host-ictv-vmr-virus-names-title-abstr...
5,viruses:e12c1ef7df4714b11631,Aastrivirus faecicola,ncbi:4530,Oryza sativa,"(""Aastrivirus faecicola""[Title/Abstract] OR ""L...",7a319130b2bd4c1da14bc6a23376ffb713ea2adf6f4fc7...,pathogen-host-ictv-vmr-virus-names-title-abstr...
6,viruses:28ffe6836272eac6c7ca,Abaestuovirus 8,ncbi:4530,Oryza sativa,"(""Abaestuovirus 8""[Title/Abstract] OR ""Inoviri...",18b17eccb1d773cd8c104ddf75a71caf0510c7f144f561...,pathogen-host-ictv-vmr-virus-names-title-abstr...
7,viruses:be644fbc773faa1b901a,Abaiavirus POA1180,ncbi:4530,Oryza sativa,"(""Abaiavirus POA1180""[Title/Abstract] OR ""Ochr...",d952e110742dbfe99f723aa4e7e239dad3643b26bb8a8a...,pathogen-host-ictv-vmr-virus-names-title-abstr...
8,viruses:51f0be9350067e48ae11,Abbeymikolonvirus abbeymikolon,ncbi:4530,Oryza sativa,"(""Abbeymikolonvirus abbeymikolon""[Title/Abstra...",9a4e4458e8e1768cdf60ff083455d496776a38b07dde49...,pathogen-host-ictv-vmr-virus-names-title-abstr...
9,viruses:62a675614c07abf300f2,Aberdnavirus waitai,ncbi:4530,Oryza sativa,"(""Aberdnavirus waitai""[Title/Abstract] OR ""Avo...",3c33c9ae78b453ca3bd0762eb229acd86cf293c36d5934...,pathogen-host-ictv-vmr-virus-names-title-abstr...


2026-11-03 00:38 INFO Generated 10 preview queries; full acquisition covers all 17554 pairs with strategy pathogen-host-ictv-vmr-virus-names-title-abstract-v3


## Step 4 · Preflight counts and acquisition readiness

In [5]:
from pathogen_atlas_v4.pubmed_client import literature_cache_key
PAIR_COUNT = len(resolved.pairs)
UNIQUE_PATHOGENS = len(resolved.pathogen_ids)
UNIQUE_HOSTS = len(resolved.host_ids)
SAMPLING = CONFIG['acquisition'].get('sampling', {})
SAMPLE_MAXIMUM = SAMPLING.get('max_pmids_per_pair', 200)
if not isinstance(SAMPLE_MAXIMUM, int) or not 1 <= SAMPLE_MAXIMUM <= 200:
    raise ValueError('Full workflow screens at most 200 abstracts per pathogen–host pair; set max_pmids_per_pair to an integer from 1 to 200.')
SAMPLE_OLDEST = int(SAMPLING.get('oldest_pmids_per_pair', 30))
SAMPLE_SEED = int(SAMPLING.get('random_seed', 42))
SAMPLE_POLICY_VERSION = str(SAMPLING.get('sample_policy_version', 'stable-prefix-hash-v1'))
# Cache identity excludes the sample cap, host selector, and pathogen selector so overlapping acquisitions are reusable.
cache_key = literature_cache_key(query_version=CONFIG['acquisition']['query_strategy_version'],
                                  pathogen_literature_query_version=CONFIG['acquisition']['pathogen_literature_query_version'],
                                  taxonomy_snapshot=TAXONOMY_ROOT.parent.name,
                                  host_expansion_hash=HOST_EXPANSION_HASH)
CACHE_DIR = ROOT / 'data/v4/cache/literature' / cache_key
CACHE_DIR.mkdir(parents=True, exist_ok=True)
preflight = {'pairs_to_search': PAIR_COUNT, 'distinct_pathogens': UNIQUE_PATHOGENS, 'distinct_hosts': UNIQUE_HOSTS,
             'query_strategy_version': CONFIG['acquisition']['query_strategy_version'],
             'pair_checkpoint': str(CACHE_DIR / 'pair_search_checkpoints.jsonl'),
             'per_pair_queries': True, 'host_expansion_hash': HOST_EXPANSION_HASH,
             'host_expansion_model': CONFIG['acquisition'].get('host_expansion_model'),
             'screen_sample_cap_per_pair': SAMPLE_MAXIMUM,
             'oldest_pmids_per_pair': SAMPLE_OLDEST, 'random_seed': SAMPLE_SEED, 'sample_policy_version': SAMPLE_POLICY_VERSION, 'network_calls_made': 0}
preflight['pubmed_alias_filter'] = ALIAS_FILTER_COUNTS
display(preflight)
LOG.info('PubMed preflight: pairs=%d pathogens=%d hosts=%d cache=%s aliases=%d searchable_aliases=%d excluded_shared=%d excluded_non_nomenclatural=%d', PAIR_COUNT, UNIQUE_PATHOGENS, UNIQUE_HOSTS, CACHE_DIR, ALIAS_FILTER_COUNTS['aliases_total'], ALIAS_FILTER_COUNTS['aliases_searchable'], ALIAS_FILTER_COUNTS['aliases_shared_across_pathogens'], ALIAS_FILTER_COUNTS['aliases_non_nomenclatural'])


{'pairs_to_search': 17554,
 'distinct_pathogens': 17554,
 'distinct_hosts': 1,
 'query_strategy_version': 'pathogen-host-ictv-vmr-virus-names-title-abstract-v3',
 'pair_checkpoint': '/Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/cache/literature/77fe35e7a12ae285cba66755aaa197029bdb4a68c6a474f5a13c8cb0646321c9/pair_search_checkpoints.jsonl',
 'per_pair_queries': True,
 'screen_sample_cap_per_pair': 200,
 'oldest_pmids_per_pair': 30,
 'random_seed': 42,
 'sample_policy_version': 'stable-prefix-hash-v1',
 'network_calls_made': 0,
 'pubmed_alias_filter': {'aliases_total': 233550,
  'aliases_missing_name': 0,
  'aliases_unmapped': 21291,
  'aliases_non_nomenclatural': 54478,
  'aliases_shared_across_pathogens': 3267,
  'aliases_vmr_records_checked': 36743,
  'aliases_vmr_terms_extracted': 65410,
  'aliases_searchable': 29251}}

2026-11-03 00:38 INFO PubMed preflight: pairs=17554 pathogens=17554 hosts=1 cache=/Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/cache/literature/77fe35e7a12ae285cba66755aaa197029bdb4a68c6a474f5a13c8cb0646321c9 aliases=233550 searchable_aliases=29251 excluded_shared=3267 excluded_non_nomenclatural=54478


## Step 5 · Search every selected pair

The client searches every selected pathogen–host pair, retains the complete PMID set, and checkpoints each result for resumability. Requests run concurrently under a shared NCBI rate limiter. Article titles and abstracts are fetched later for the stable sample selected by the run configuration. Search result sets above PubMed’s 10,000-record UID retrieval ceiling are segmented by publication date; a segment that still cannot be retrieved completely raises an error instead of silently truncating.

In [6]:
import os
from pathogen_atlas_v4.pubmed_client import acquire_pubmed_pairs, ncbi_api_key

NCBI_EMAIL = ncbi_contact_email(ROOT)
NCBI_API_KEY = ncbi_api_key()
LOG.info('Starting full PubMed acquisition: pairs=%d api_key_configured=%s cache=%s', len(resolved.pairs), bool(NCBI_API_KEY), CACHE_DIR)
acquisition = acquire_pubmed_pairs(
    resolved.pairs, pathogens, hosts, ALIASES, CACHE_DIR,
    query_version=CONFIG['acquisition']['query_strategy_version'],
    host_search_expansions=HOST_SEARCH_EXPANSIONS,
    email=NCBI_EMAIL, api_key=NCBI_API_KEY, logger=LOG,
    sample_maximum=SAMPLE_MAXIMUM, sample_oldest_count=SAMPLE_OLDEST, sample_seed=SAMPLE_SEED,
    max_workers=CONFIG['acquisition'].get('max_concurrent_requests', 8),
)
display(acquisition)
LOG.info('Pair PubMed acquisition complete: %s', acquisition)
LOG.info('Pathogen-only total-publication counts are deferred until pathogen–host screening identifies the accepted pathogen set.')


2026-11-03 00:38 INFO Starting full PubMed acquisition: pairs=17554 api_key_configured=True cache=/Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/cache/literature/77fe35e7a12ae285cba66755aaa197029bdb4a68c6a474f5a13c8cb0646321c9


Using the configured Git email as the NCBI contact address.


2026-11-03 00:38 INFO PubMed pair-search alias safeguard: {"aliases_missing_name": 0, "aliases_non_nomenclatural": 54478, "aliases_searchable": 29251, "aliases_shared_across_pathogens": 3267, "aliases_total": 233550, "aliases_unmapped": 21291, "aliases_vmr_records_checked": 36743, "aliases_vmr_terms_extracted": 65410}


2026-11-03 00:38 INFO PubMed pair acquisition settings: pairs=17554 already_checkpointed=0 searches_to_run=17554; sample cap=200 oldest=30 seed=42; workers=8; API key=True; cached article PMIDs=0


2026-11-03 00:39 WARNING ESearch retry 1/6 after HTTPError; sleeping 1s


2026-11-03 00:39 WARNING ESearch retry 1/6 after HTTPError; sleeping 1s


... (822 outputs left out)

2026-11-03 01:15 INFO PubMed pair checkpoints 17554/17554 (100.0%); updated this session=17554; ETA 0m


{'pairs_total': 17554,
 'pairs_complete': 17554,
 'pairs_updated_this_session': 17554,
 'pair_matches': 6452,
 'sample_selected_matches': 4589,
 'sample_cap_per_pair': 200,
 'oldest_per_pair': 30,
 'sample_seed': 42,
 'unique_sample_articles_cached': 0,
 'output_dir': '/Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/cache/literature/77fe35e7a12ae285cba66755aaa197029bdb4a68c6a474f5a13c8cb0646321c9/tables',
 'checkpoint': '/Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/cache/literature/77fe35e7a12ae285cba66755aaa197029bdb4a68c6a474f5a13c8cb0646321c9/pair_search_checkpoints.jsonl'}

2026-11-03 01:16 INFO Pair PubMed acquisition complete: {'pairs_total': 17554, 'pairs_complete': 17554, 'pairs_updated_this_session': 17554, 'pair_matches': 6452, 'sample_selected_matches': 4589, 'sample_cap_per_pair': 200, 'oldest_per_pair': 30, 'sample_seed': 42, 'unique_sample_articles_cached': 0, 'output_dir': '/Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/cache/literature/77fe35e7a12ae285cba66755aaa197029bdb4a68c6a474f5a13c8cb0646321c9/tables', 'checkpoint': '/Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/cache/literature/77fe35e7a12ae285cba66755aaa197029bdb4a68c6a474f5a13c8cb0646321c9/pair_search_checkpoints.jsonl'}


2026-11-03 01:16 INFO Pathogen-only total-publication counts are deferred until pathogen–host screening identifies the accepted pathogen set.
